In [ ]:
import os
import glob

import torch

from monai.data import Dataset, DataLoader, CacheDataset

#設定資料路徑
data_dir = "../data/raw/Task01_BrainTumour"

images_dir = os.path.join(data_dir, "imagesTr")
labels_dir = os.path.join(data_dir, "labelsTr")

#讀取資料集(用glob)
images = sorted(
    glob.glob(
        os.path.join(images_dir,"*.nii.gz")
    )
)

labels = sorted(
    glob.glob(
        os.path.join(labels_dir,"*.nii.gz")
    )
)
#查看數量（Task01 BrainTumour 約 484 cases）
print(len(images))
print(len(labels))

484
484


In [2]:
data_dicts = [
    {
        "image": image_name,
        "label": label_name
    }
    for image_name, label_name in zip(images, labels)
]

data_dicts[:2]

[{'image': '../data/raw/Task01_BrainTumour/imagesTr/BRATS_001.nii.gz',
  'label': '../data/raw/Task01_BrainTumour/labelsTr/BRATS_001.nii.gz'},
 {'image': '../data/raw/Task01_BrainTumour/imagesTr/BRATS_002.nii.gz',
  'label': '../data/raw/Task01_BrainTumour/labelsTr/BRATS_002.nii.gz'}]

In [3]:
train_files = data_dicts[:400]
val_files = data_dicts[400:]

print(len(train_files))
print(len(val_files))

400
84


In [4]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    NormalizeIntensityd
)

train_transform = Compose(
[
    LoadImaged(
        keys=["image","label"]
    ),

    EnsureChannelFirstd(
        keys=["image","label"]
    ),

    Orientationd(
        keys=["image","label"],
        axcodes="RAS"
    ),

    Spacingd(
        keys=["image","label"],
        pixdim=(1,1,1),
        mode=("bilinear","nearest")
    ),

    NormalizeIntensityd(
        keys="image"
    )
]
)

mode="nearest"

/opt/anaconda3/envs/medical-ai/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [5]:
CacheDataset
train_ds = CacheDataset(
    data=train_files,
    transform=train_transform,
    cache_rate=1.0,
    num_workers=4
)

/opt/anaconda3/envs/medical-ai/lib/python3.11/site-packages/monai/data/dataset.py:911: UserWarning: tqdm is not installed, will not show the caching progress bar.
  warnings.warn("tqdm is not installed, will not show the caching progress bar.")


In [6]:
train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=2
)

In [7]:
batch = next(iter(train_loader))
batch.keys()

dict_keys(['image', 'label'])

In [8]:
print(batch["image"].shape)
print(batch["label"].shape)

torch.Size([1, 4, 240, 240, 155])
torch.Size([1, 1, 240, 240, 155])


In [9]:
batch["image"]

metatensor([[[[[-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           [-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           [-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           ...,
           [-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           [-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           [-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757]],

          [[-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           [-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           [-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           ...,
           [-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           [-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           [-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757]],

          [[-0.3757, -0.3757, -0.3757,  ..., -0.3757, -0.3757, -0.3757],
           

In [10]:
batch = next(iter(train_loader))
print(batch["image"].shape)

torch.Size([1, 4, 240, 240, 155])
